In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
os.chdir("../")

In [3]:
from ease_recommender import *
from npmi_recommender import *

import pickle as p

In [4]:
def create_mat(row, col, bool_to_int=True):
    # bool_to_int won't count duplicates in the same row, creates a different weighting basically
    if bool_to_int:
        data = np.ones_like(row, dtype=bool)
        return csr_matrix((data, (row, col))).astype(np.int64)
    else:
        data = np.ones_like(row, dtype=np.int64)
        return csr_matrix((data, (row, col)))

def check_if_all_terms_in_str(q, terms):
    for term in terms:
        if term not in q:
            return False

    return True

def get_cat2idx(category_type, D):
    if category_type == "track":
        return D["track2idx"]
    elif category_type == "album":
        return D["album2idx"]
    elif category_type == "artist":
        return D["artist2idx"]
    else:
        raise NotImplementedError

def find_match_using_terms(terms, cat2idx):
    matches = []
    for name in cat2idx.keys():
        if check_if_all_terms_in_str(name, terms):
            matches.append(name)

    if len(matches) > 1:
        raise Exception("Multiple matches found, filter down to a single match", matches)

    return matches[0]

In [5]:
print("loading cache data...")
D = p.load(open("cached_data/spotify_preprocessed.p", "rb"))

print("building csr matrices...")

# TODO: finish implementing track and album level recommendations

# track_mat = create_mat(D["playlist_indices"], D["track_indices"])
# album_mat = create_mat(D["playlist_indices"], D["album_indices"])
artist_mat = create_mat(D["playlist_indices"], D["artist_indices"])

print("done")

loading cache data...
building csr matrices...
done


In [116]:
cat2idx = get_cat2idx("artist", D)
idx2cat = {v:k for k, v in cat2idx.items()}

In [117]:
# use two items that you believe are similar to optimize the value of lambda_

a_name = find_match_using_terms(["sgeir", "7xUZ4069zcyBM4Bn10NQ1c"], cat2idx)
# a_name = find_match_using_terms(["7fNWySjsDn74LCawyJ27EQ"], cat2idx)

a = cat2idx[a_name]
print(f"{a_name=}")
print(f"Num Rows: {artist_mat[:, a].sum()}")

# a = cat2idx[find_match_using_terms(["Fleet Foxes"], cat2idx)]

# b = cat2idx[find_match_using_terms(["Fleet Foxes"], cat2idx)]
# b = cat2idx[find_match_using_terms(["Bon Iver", "4LEiUm1SRbFMgfqnQTwUbQ"], cat2idx)]
# b = cat2idx[find_match_using_terms(["SOHN"], cat2idx)]

# b_name = find_match_using_terms(["7fNWySjsDn74LCawyJ27EQ"], cat2idx)
b_name = find_match_using_terms(["Highas"], cat2idx)
b = cat2idx[b_name]
print(f"{b_name=}")
print(f"Num Rows: {artist_mat[:, b].sum()}")

a_name='Ásgeir (spotify:artist:7xUZ4069zcyBM4Bn10NQ1c)'
Num Rows: 1981
b_name='Highasakite (spotify:artist:5awQWdBpLqN2KFVRN8w56T)'
Num Rows: 412


In [118]:
c_name = find_match_using_terms(["Gabrielle (spotify:artist:4OovmAu23KrDlDQI2UbneL)"], cat2idx)
c = cat2idx[c_name]
print(f"{c_name=}")
print(f"Num Rows: {artist_mat[:, c].sum()}")

c_name='Gabrielle (spotify:artist:4OovmAu23KrDlDQI2UbneL)'
Num Rows: 71


In [9]:
import numpy as np
from scipy import sparse

def calculate_pmi_matrix(co_counts, alpha=1.0):
    """
    Calculates Alpha-Smoothed PMI for the entire co-occurrence matrix.
    Only computes values for observed entries (non-zeros) to maintain sparsity.
    
    Formula:
    PMI(x,y) = log( (C_xy + alpha) * N_smoothed / (S_x * S_y) )
             = log(C_xy + alpha) + log(N_smoothed) - log(S_x) - log(S_y)
    
    Args:
        co_counts: (n_items, n_items) scipy.sparse matrix (Counts of x, y).
        alpha: Smoothing parameter.
        
    Returns:
        pmi_matrix: (n_items, n_items) scipy.sparse matrix containing PMI values.
    """
    # Ensure we work with a COO matrix to easily access .row, .col, and .data
    # This is a cheap conversion if it's already CSR/CSC
    co_counts_coo = co_counts.tocoo()
    
    n_items = co_counts_coo.shape[0]
    
    # 1. Global Constants
    # Total sum of all counts in the matrix
    N_raw = co_counts_coo.sum()
    
    # Smoothed Normalization Factor: N + alpha * V^2
    N_smoothed = N_raw + (alpha * (n_items ** 2))
    
    # 2. Calculate Smoothed Marginals (S_x and S_y)
    # Row sums gives us C_x (raw marginal counts)
    # We flatten it to a dense 1D array for easy lookup
    C_marginal = np.array(co_counts_coo.sum(axis=1)).flatten()
    
    # Apply smoothing to marginals: C_x + alpha * V
    S_marginal = C_marginal + (alpha * n_items)
    
    # 3. Vectorized PMI Calculation on Non-Zero Elements
    # We operate directly on the 'data' array of the sparse matrix.
    
    # rows and cols indices for every non-zero entry
    rows = co_counts_coo.row
    cols = co_counts_coo.col
    
    # Data is C_xy
    C_xy = co_counts_coo.data
    
    # Look up the marginals for every interaction
    # If interaction is (item 5, item 99), we grab S_marginal[5] and S_marginal[99]
    Sx_vals = S_marginal[rows]
    Sy_vals = S_marginal[cols]
    
    # Apply the Log-PMI formula decomposed:
    # log(C_xy + alpha) + log(N_smoothed) - log(Sx) - log(Sy)
    
    term1 = np.log(C_xy + alpha)
    term2 = np.log(N_smoothed)
    term3 = np.log(Sx_vals)
    term4 = np.log(Sy_vals)
    
    pmi_data = term1 + term2 - term3 - term4
    
    # 4. Reconstruct Sparse Matrix
    # We create a new CSR matrix with the calculated PMI values
    pmi_matrix = sparse.csr_matrix(
        (pmi_data, (rows, cols)), 
        shape=co_counts_coo.shape
    )
    
    return pmi_matrix

In [10]:
mat = artist_mat
mat = csr_array(mat)

In [11]:
mat.shape

(1000000, 295860)

In [12]:
X = mat.T @ mat

In [13]:
X.shape

(295860, 295860)

In [14]:
import numpy as np
from scipy import sparse

def sparse_laplace_sppmi(X, alpha=1.0, normalize=False, zero_diag=True, non_neg=False):
    # Ensure input is CSR for fast row operations
    if not sparse.isspmatrix_csr(X):
        X = X.tocsr()
        
    # 1. Get Geometry of the Data
    # V: Vocabulary size (rows/cols)
    rows, cols = X.shape
    
    # N_raw: Total real observations
    N_raw = X.sum()
    
    # 2. Calculate "Smoothed" Marginals (The Global Statistics)
    # We pretend we added alpha to every cell, but we compute the sums analytically.
    
    # Virtual Total N = Real N + (alpha * Total Possible Cells)
    N_smoothed = N_raw + (alpha * rows * cols)
    
    # Raw marginals (sum of rows/cols)
    row_sums_raw = np.array(X.sum(axis=1)).flatten()
    col_sums_raw = np.array(X.sum(axis=0)).flatten()
    
    # Smoothed marginals = Raw Sum + (alpha * row_length)
    # Each row has 'cols' number of cells, so we add alpha * cols to the row sum
    P_x = (row_sums_raw + (alpha * cols)) / N_smoothed
    P_y = (col_sums_raw + (alpha * rows)) / N_smoothed
    
    # 3. Operate ONLY on Non-Zero Data (The Sparse Trick)
    # We extract the indices of existing data points to calculate their new PMI
    # efficiently, skipping the billions of zeros.
    
    # Create a copy to store results
    sppmi = X.copy().astype(np.float32)
    
    # Get indices of non-zero elements
    row_indices, col_indices = X.nonzero()
    
    # Get the raw counts
    raw_counts = np.array(X.data)
    
    # Smooth the counts: Count_new = Count_raw + alpha
    smoothed_counts = raw_counts + alpha
    
    # Calculate P(x,y) for these specific entries
    P_xy = smoothed_counts / N_smoothed
    
    # 4. Vectorized PMI Calculation
    # PMI = log( P(x,y) / (P(x) * P(y)) )
    # Note: P_x[row_indices] grabs the specific P(x) for every non-zero entry
    
    # Numerator is P_xy
    # Denominator is P(x) * P(y)
    denominator = P_x[row_indices] * P_y[col_indices]
    
    # Calculate PMI (using log2)
    pmi_values = np.log2(P_xy / denominator)
    
    if normalize:
        pmi_values = pmi_values / -np.log2(P_xy)
    
    # 7. Update the matrix data
    sppmi.data = pmi_values
    
    if non_neg:
        sppmi.data = np.where(sppmi.data > 0, sppmi.data, 0)
    
    if zero_diag:
        sppmi.setdiag(np.zeros(sppmi.shape[0]))
    
    # 8. Clean up (Remove explicit zeros to keep matrix sparse)
    sppmi.eliminate_zeros()
    
    return sppmi

In [15]:
alpha = .1 * .85

In [16]:
sppmi = sparse_laplace_sppmi(X, alpha=alpha, non_neg=True)

In [17]:
metric = (np.argsort(-sppmi[:, a].toarray()).tolist().index(b) + np.argsort(-sppmi[:, b].toarray()).tolist().index(a))/2
metric

10.0

In [235]:
def generate_ranking_score(a, ranking_type):
    px = mat.mean(axis=0)
    
    px_given_y = (mat[:, a:a+1] * mat).sum(axis=0) / mat[:, a].sum()
    
    lift = px_given_y / px
    
    lift_normalizer = 1/np.where(px < px[a], px, px[a])
    
    if ranking_type == "px":
        return px
    elif ranking_type == "px_given_y":
        return px_given_y
    elif ranking_type == "npmi_alt":
        return np.log2(lift) / np.log2(lift_normalizer)
    elif ranking_type == "npmi":
        return np.log2(lift) / -np.log2(px_given_y)
    elif ranking_type == "lift":
        return lift
    elif ranking_type == "weighted_lift":
        return lift * px_given_y
    elif ranking_type == "normalized_lift":
        return lift / lift_normalizer
    elif ranking_type == "normalized_weighted_lift":
        return (lift / lift_normalizer) * px_given_y
    else:
        raise NotImplementedError

In [236]:
from tqdm.auto import tqdm

for ranking_type in tqdm(["px", "px_given_y", "npmi_alt", "npmi", "lift", "weighted_lift", "normalized_lift", "normalized_weighted_lift"]):
    metric = (np.argsort(-generate_ranking_score(a, ranking_type)).tolist().index(b) +
                np.argsort(-generate_ranking_score(b, ranking_type)).tolist().index(a))/2
    
    print(metric)

  0%|          | 0/8 [00:00<?, ?it/s]

5559.0
495.0


C:\Users\johns\AppData\Local\Temp\ipykernel_17924\1428710982.py:15: RuntimeWarning: divide by zero encountered in log2
  return np.log2(lift) / np.log2(lift_normalizer)


402.5


C:\Users\johns\AppData\Local\Temp\ipykernel_17924\1428710982.py:17: RuntimeWarning: divide by zero encountered in log2
  return np.log2(lift) / -np.log2(px_given_y)
C:\Users\johns\AppData\Local\Temp\ipykernel_17924\1428710982.py:17: RuntimeWarning: divide by zero encountered in divide
  return np.log2(lift) / -np.log2(px_given_y)
C:\Users\johns\AppData\Local\Temp\ipykernel_17924\1428710982.py:17: RuntimeWarning: invalid value encountered in divide
  return np.log2(lift) / -np.log2(px_given_y)


63.5
2132.0
127.5
187.5
284.0


In [237]:
# score = generate_ranking_score(b, "normalized_weighted_lift")
# score = generate_ranking_score(b, "normalized_lift")
# score = generate_ranking_score(b, "weighted_lift")
score = generate_ranking_score(b, "npmi")
# score = generate_ranking_score(b, "npmi_alt")

print(np.argsort(-score).tolist().index(a))

13


C:\Users\johns\AppData\Local\Temp\ipykernel_17924\1428710982.py:17: RuntimeWarning: divide by zero encountered in log2
  return np.log2(lift) / -np.log2(px_given_y)
C:\Users\johns\AppData\Local\Temp\ipykernel_17924\1428710982.py:17: RuntimeWarning: divide by zero encountered in divide
  return np.log2(lift) / -np.log2(px_given_y)
C:\Users\johns\AppData\Local\Temp\ipykernel_17924\1428710982.py:17: RuntimeWarning: invalid value encountered in divide
  return np.log2(lift) / -np.log2(px_given_y)


In [238]:
top_k = 10

for i in np.argsort(-score)[:top_k]:
    print(idx2cat[i])

Hanzee (spotify:artist:5yM1po4NHvE2yE1Kf84LWJ)
Nils Bech (spotify:artist:57QhXfAsLsIRtgC1VfHu1F)
Rockettothesky (spotify:artist:0nu7qEOc8X8UFK10d8lsLw)
Elsa & Emilie (spotify:artist:4HDNQLqhooVfWXtIRMyqMY)
Ganic (spotify:artist:2qT4UNxM0aOz52gpSXuooT)
Feitn Fra Kolbotn (spotify:artist:5pTo5SvSAVsfJ9vbVwQyUt)
Gabrielle (spotify:artist:4OovmAu23KrDlDQI2UbneL)
ZL-Project (spotify:artist:4w8PGLhS3yzYSzeV3x2hkA)
Kjartan Lauritzen (spotify:artist:0TW5M8RYADmgeCP1q523hf)
Surferosa (spotify:artist:5WUYimkWakv41ZVVIq3BDr)


In [208]:
score = generate_ranking_score(a, "npmi")

np.argsort(-score).tolist().index(b)

C:\Users\johns\AppData\Local\Temp\ipykernel_17924\3250455684.py:13: RuntimeWarning: divide by zero encountered in log2
  return np.log2(lift) / -np.log2(px_given_y)
C:\Users\johns\AppData\Local\Temp\ipykernel_17924\3250455684.py:13: RuntimeWarning: divide by zero encountered in divide
  return np.log2(lift) / -np.log2(px_given_y)
C:\Users\johns\AppData\Local\Temp\ipykernel_17924\3250455684.py:13: RuntimeWarning: invalid value encountered in divide
  return np.log2(lift) / -np.log2(px_given_y)


114

In [211]:
score = generate_ranking_score(b, "npmi_alt")

np.argsort(-score).tolist().index(a)

C:\Users\johns\AppData\Local\Temp\ipykernel_17924\3250455684.py:11: RuntimeWarning: divide by zero encountered in log2
  return np.log2(lift) / np.log2(lift_normalizer)


753

In [212]:
top_k = 10

for i in np.argsort(-score)[:top_k]:
    print(idx2cat[i])

Highasakite (spotify:artist:5awQWdBpLqN2KFVRN8w56T)
Hanzee (spotify:artist:5yM1po4NHvE2yE1Kf84LWJ)
Nils Bech (spotify:artist:57QhXfAsLsIRtgC1VfHu1F)
Rockettothesky (spotify:artist:0nu7qEOc8X8UFK10d8lsLw)
Elsa & Emilie (spotify:artist:4HDNQLqhooVfWXtIRMyqMY)
Ganic (spotify:artist:2qT4UNxM0aOz52gpSXuooT)
Feitn Fra Kolbotn (spotify:artist:5pTo5SvSAVsfJ9vbVwQyUt)
Gabrielle (spotify:artist:4OovmAu23KrDlDQI2UbneL)
ZL-Project (spotify:artist:4w8PGLhS3yzYSzeV3x2hkA)
Kjartan Lauritzen (spotify:artist:0TW5M8RYADmgeCP1q523hf)


In [18]:
X.shape

(295860, 295860)

In [19]:
px = mat.mean(axis=0)

In [24]:
mat.shape

(1000000, 295860)

In [39]:
px_given_y = (mat[:, a:a+1] * mat).sum(axis=0) / mat[:, a].sum()

In [45]:
lift = px_given_y / px

In [48]:
top_k = 10

for i in np.argsort(-lift)[:top_k]:
    print(idx2cat[i])

Ásgeir (spotify:artist:7xUZ4069zcyBM4Bn10NQ1c)
Malachi Jackson (spotify:artist:6rfzWpkjEVWygReumyxJzc)
Jiggabits (spotify:artist:0Jw1CUzPv1WBB8uOUCj9yU)
Gilus (spotify:artist:50wkBXgHtMxRA905sYlUxP)
Ice Cream Cathedral (spotify:artist:4b75XvQ2tB53X7GoD5Cvhn)
New West Guitar Group (spotify:artist:6eGQni9pk1y5tsiZ5TBBID)
Jake Coronado (spotify:artist:5VW29S4YpWMDvohHbqUOLF)
Tuska (spotify:artist:3RjhhylIPk66I6HIuTUG5b)
Similia (spotify:artist:4TNlBKqH3ujY7r6fOrsJZx)
Robert Mitchell 3io (spotify:artist:05oAeVFNUPHOqBXDx19GNA)


In [49]:
weighted_lift = lift * px_given_y

In [50]:
top_k = 10

for i in np.argsort(-weighted_lift)[:top_k]:
    print(idx2cat[i])

Ásgeir (spotify:artist:7xUZ4069zcyBM4Bn10NQ1c)
James Vincent McMorrow (spotify:artist:7FDlvgcodNfC0IBdWevl4u)
José González (spotify:artist:6xrCU6zdcSTsG2hLrojpmI)
Sylvan Esso (spotify:artist:39vA9YljbnOApXKniLWBZv)
Ben Howard (spotify:artist:5schNIzWdI9gJ1QRK8SBnc)
Bon Iver (spotify:artist:4LEiUm1SRbFMgfqnQTwUbQ)
alt-J (spotify:artist:3XHO7cRUPCLOr6jwp8vsx5)
Dustin Tebbutt (spotify:artist:0z9hynUsIjf0ddI4uHqPWX)
Chet Faker (spotify:artist:2Q0MyH5YMI5HPQjFjlq5g3)
SOHN (spotify:artist:6XZYAWJLL8UIbxAqjKj3cg)


In [58]:
lift_normalizer = 1/np.where(px < px[a], px, px[a])

normalized_lift = lift / lift_normalizer

In [59]:
top_k = 10

for i in np.argsort(-normalized_lift)[:top_k]:
    print(idx2cat[i])

Ásgeir (spotify:artist:7xUZ4069zcyBM4Bn10NQ1c)
Dustin Tebbutt (spotify:artist:0z9hynUsIjf0ddI4uHqPWX)
Volcano Choir (spotify:artist:6gAtOqhriLzOzb3Qqmg5kO)
Lo-Fang (spotify:artist:5EDkJDlRNcMs3ewliB24QA)
Novo Amor (spotify:artist:0rZp7G3gIH6WkyeXbrZnGi)
PHOX (spotify:artist:3ix4iw2URncSdE7X292bXy)
SOHN (spotify:artist:6XZYAWJLL8UIbxAqjKj3cg)
The Kite String Tangle (spotify:artist:3D6cosC5ZOLCpRxt6T3XS7)
Nick Mulvey (spotify:artist:3x8FbPjh2Qz55XMdE2Yalj)
Vancouver Sleep Clinic (spotify:artist:77BznF1Dr1k5KyEZ6Nn3jB)


In [62]:
normalized_weighted_lift = normalized_lift * px_given_y

In [63]:
top_k = 10

for i in np.argsort(-normalized_weighted_lift)[:top_k]:
    print(idx2cat[i])

Ásgeir (spotify:artist:7xUZ4069zcyBM4Bn10NQ1c)
James Vincent McMorrow (spotify:artist:7FDlvgcodNfC0IBdWevl4u)
José González (spotify:artist:6xrCU6zdcSTsG2hLrojpmI)
Sylvan Esso (spotify:artist:39vA9YljbnOApXKniLWBZv)
Ben Howard (spotify:artist:5schNIzWdI9gJ1QRK8SBnc)
Bon Iver (spotify:artist:4LEiUm1SRbFMgfqnQTwUbQ)
alt-J (spotify:artist:3XHO7cRUPCLOr6jwp8vsx5)
Dustin Tebbutt (spotify:artist:0z9hynUsIjf0ddI4uHqPWX)
Chet Faker (spotify:artist:2Q0MyH5YMI5HPQjFjlq5g3)
SOHN (spotify:artist:6XZYAWJLL8UIbxAqjKj3cg)


In [188]:
# npmi_alt = lift / lift_normalizer
npmi_alt = np.log2(lift) / np.log2(lift_normalizer)

top_k = 10

for i in np.argsort(-npmi_alt)[:top_k]:
    print(idx2cat[i])

Ásgeir (spotify:artist:7xUZ4069zcyBM4Bn10NQ1c)
Dustin Tebbutt (spotify:artist:0z9hynUsIjf0ddI4uHqPWX)
Ásgeir Trausti (spotify:artist:7fNWySjsDn74LCawyJ27EQ)
Allman Brown (spotify:artist:239Y6QdFqVFfdsw6moqSEN)
Low Volts (spotify:artist:3PxUwSSsVaW0XyBiRJF2oS)
PHOX (spotify:artist:3ix4iw2URncSdE7X292bXy)
Volcano Choir (spotify:artist:6gAtOqhriLzOzb3Qqmg5kO)
Lo-Fang (spotify:artist:5EDkJDlRNcMs3ewliB24QA)
Nick Mulvey (spotify:artist:3x8FbPjh2Qz55XMdE2Yalj)
Novo Amor (spotify:artist:0rZp7G3gIH6WkyeXbrZnGi)


C:\Users\johns\AppData\Local\Temp\ipykernel_17924\3717324443.py:2: RuntimeWarning: divide by zero encountered in log2
  npmi_alt = np.log2(lift) / np.log2(lift_normalizer)


In [189]:
np.argsort(-npmi_alt).tolist().index(b)

52

In [175]:
npmi_alt = np.log2(lift) / np.log2(lift_normalizer)
# npmi_alt = np.log2(lift) / np.log2(1/lift_normalizer)

top_k = 10

for i in np.argsort(-npmi_alt)[:top_k]:
    print(idx2cat[i])

Ásgeir (spotify:artist:7xUZ4069zcyBM4Bn10NQ1c)
Dustin Tebbutt (spotify:artist:0z9hynUsIjf0ddI4uHqPWX)
Ásgeir Trausti (spotify:artist:7fNWySjsDn74LCawyJ27EQ)
Allman Brown (spotify:artist:239Y6QdFqVFfdsw6moqSEN)
Low Volts (spotify:artist:3PxUwSSsVaW0XyBiRJF2oS)
PHOX (spotify:artist:3ix4iw2URncSdE7X292bXy)
Volcano Choir (spotify:artist:6gAtOqhriLzOzb3Qqmg5kO)
Lo-Fang (spotify:artist:5EDkJDlRNcMs3ewliB24QA)
Nick Mulvey (spotify:artist:3x8FbPjh2Qz55XMdE2Yalj)
Novo Amor (spotify:artist:0rZp7G3gIH6WkyeXbrZnGi)


C:\Users\johns\AppData\Local\Temp\ipykernel_17924\2741643677.py:1: RuntimeWarning: divide by zero encountered in log2
  npmi_alt = np.log2(lift) / np.log2(lift_normalizer)


In [176]:
np.argsort(-npmi_alt).tolist().index(b)

52

In [135]:
npmi = np.log2(lift) / -np.log2(px_given_y)

top_k = 10

for i in np.argsort(-npmi)[:top_k]:
    print(idx2cat[i])

James Vincent McMorrow (spotify:artist:7FDlvgcodNfC0IBdWevl4u)
Ben Howard (spotify:artist:5schNIzWdI9gJ1QRK8SBnc)
alt-J (spotify:artist:3XHO7cRUPCLOr6jwp8vsx5)
Bon Iver (spotify:artist:4LEiUm1SRbFMgfqnQTwUbQ)
Sylvan Esso (spotify:artist:39vA9YljbnOApXKniLWBZv)
José González (spotify:artist:6xrCU6zdcSTsG2hLrojpmI)
Chet Faker (spotify:artist:2Q0MyH5YMI5HPQjFjlq5g3)
Hozier (spotify:artist:2FXC3k01G6Gw61bmprjgqS)
Glass Animals (spotify:artist:4yvcSjfu4PC0CYQyLy4wSq)
James Blake (spotify:artist:53KwLdlmrlCelAZMaLVZqU)


C:\Users\johns\AppData\Local\Temp\ipykernel_17924\876563672.py:1: RuntimeWarning: divide by zero encountered in log2
  npmi = np.log2(lift) / -np.log2(px_given_y)
C:\Users\johns\AppData\Local\Temp\ipykernel_17924\876563672.py:1: RuntimeWarning: divide by zero encountered in divide
  npmi = np.log2(lift) / -np.log2(px_given_y)
C:\Users\johns\AppData\Local\Temp\ipykernel_17924\876563672.py:1: RuntimeWarning: invalid value encountered in divide
  npmi = np.log2(lift) / -np.log2(px_given_y)


In [136]:
np.argsort(-npmi).tolist().index(b)

114

In [75]:
np.argsort(-lift).tolist().index(b)

2330

In [76]:
np.argsort(-weighted_lift).tolist().index(b)

134

In [77]:
np.argsort(-normalized_lift).tolist().index(b)

344

In [78]:
np.argsort(-normalized_weighted_lift).tolist().index(b)

566

In [80]:
from scipy.optimize import minimize_scalar

In [84]:
np.exp(np.log(px_given_y) * x) * normalized_lift

C:\Users\johns\AppData\Local\Temp\ipykernel_17924\1504623455.py:1: RuntimeWarning: divide by zero encountered in log
  np.exp(np.log(px_given_y) * x) * normalized_lift


NameError: name 'x' is not defined

In [ ]:
from scipy.optimize import minimize

In [125]:
def f(p):
#     ranking_metric = np.exp(np.log(px_given_y) * x) * normalized_lift
#     ranking_metric = np.exp(np.log(px_given_y) * x) * lift
#     ranking_metric = px_given_y**x * lift
#     ranking_metric = px_given_y**x * lift * (1/lift_normalizer)**y
    ranking_metric = np.exp(np.log(px) * p[0] + np.log(px_given_y) * p[1] + np.log(lift_normalizer) * p[2])
    
    return np.argsort(-ranking_metric).tolist().index(b)

# res = minimize_scalar(f)
res = minimize(f, [-1, 1, -1], method="Powell")
# res = minimize(f, [-1, 1, -1], method="Nelder-Mead")

res

C:\Users\johns\AppData\Local\Temp\ipykernel_17924\3828707080.py:6: RuntimeWarning: divide by zero encountered in log
  ranking_metric = np.exp(np.log(px) * p[0] + np.log(px_given_y) * p[1] + np.log(lift_normalizer) * p[2])


 message: Optimization terminated successfully.
 success: True
  status: 0
     fun: 36
       x: [-1.577e+00  9.978e-01 -1.080e+00]
     nit: 4
   direc: [[ 1.000e+00  0.000e+00  0.000e+00]
           [ 0.000e+00  1.000e+00  0.000e+00]
           [ 0.000e+00  0.000e+00  1.000e+00]]
    nfev: 312

In [123]:
res.x

array([-1.57702903,  0.99782948, -1.07984841])

In [121]:
f([-1, 1, -1])

C:\Users\johns\AppData\Local\Temp\ipykernel_17924\3487729486.py:6: RuntimeWarning: divide by zero encountered in log
  ranking_metric = np.exp(np.log(px) * p[0] + np.log(px_given_y) * p[1] * np.log(lift_normalizer) * p[2])


293862

In [97]:
res.x

array([-0.99476447,  0.99770523])

In [109]:
1/(px_given_y**0.99476447)

C:\Users\johns\AppData\Local\Temp\ipykernel_17924\3665458016.py:1: RuntimeWarning: divide by zero encountered in divide
  1/(px_given_y**0.99476447)


array([213.98187753, 128.73295547,  19.3106036 , ...,          inf,
                inf,          inf], shape=(295860,))

In [105]:
px_given_y**-0.99476447

C:\Users\johns\AppData\Local\Temp\ipykernel_17924\3173790207.py:1: RuntimeWarning: divide by zero encountered in power
  px_given_y**-0.99476447


array([213.98187753, 128.73295547,  19.3106036 , ...,          inf,
                inf,          inf], shape=(295860,))

In [111]:
# ranking_metric = lift * (px_given_y) * (1/lift_normalizer)
ranking_metric = 1/(px_given_y**0.99476447) * lift * (1/lift_normalizer)**0.99770523

np.argsort(-ranking_metric).tolist().index(b)

C:\Users\johns\AppData\Local\Temp\ipykernel_17924\3998867154.py:2: RuntimeWarning: divide by zero encountered in divide
  ranking_metric = 1/(px_given_y**0.99476447) * lift * (1/lift_normalizer)**0.99770523
C:\Users\johns\AppData\Local\Temp\ipykernel_17924\3998867154.py:2: RuntimeWarning: invalid value encountered in multiply
  ranking_metric = 1/(px_given_y**0.99476447) * lift * (1/lift_normalizer)**0.99770523


27

In [70]:
# import pandas as pd

# pd.DataFrame({
#     "a": normalized_weighted_lift,
#     "b": weighted_lift
# }).corr("kendall")["a"]["b"]

In [72]:
# import pandas as pd

# pd.DataFrame({
#     "a": normalized_lift,
#     "b": lift
# }).corr("kendall")["a"]["b"]

In [74]:
# import pandas as pd

# pd.DataFrame({
#     "a": weighted_lift,
#     "b": lift
# }).corr("kendall")["a"]["b"]